# Native data files: build a scientific knowledge graph

Read JSON concepts and CSV relationships using only Python's standard library. Turn uncertain
external data into validated domain objects, then assemble and query a custom knowledge graph.

**Lecture 2 · Python Foundations II · CMOR 438 / INDE 577**

## Python is the lesson; scientific connections are the setting

Our graph connects mathematical foundations to scientific models and machine-learning methods:

```text
scientific measurement → vector → data matrix → matrix
entropy → probability distribution → scientific measurement
diffusion model → Markov chain → probability distribution
```

The arrows are data—not Python code. Python will give those records structure and behavior. This is
the central move in data engineering: **parse at the boundary, validate once, and let the rest of the
program work with trustworthy objects.**

## How to use this notebook

**Estimated time:** 65 minutes core, plus 40 minutes of practice and extension.

**Prerequisites:** paths, strings, lists, dictionaries, loops, functions, exceptions, classes,
dataclasses, and type hints.

Run from top to bottom in the **Rice DSM** kernel. Pause whenever the representation changes:

```text
file path → text stream → parsed dictionaries → validated objects → knowledge graph
```

No third-party package, database, network connection, or graph library is required.

## Learning objectives

By the end, you should be able to:

- construct cross-platform paths with `pathlib`;
- explain why JSON and CSV fields become different Python types;
- load JSON safely and stream CSV rows with context managers;
- distinguish syntax validation, schema validation, and domain validation;
- convert mappings into immutable, typed domain objects;
- design a graph class with explicit invariants and a small public interface;
- enforce referential integrity between nodes and relationships;
- query neighbors and find directed paths with breadth-first search;
- attach file and row context to useful exceptions; and
- serialize derived data without overwriting raw inputs.

## Why this matters in industry

A production knowledge graph might connect papers, experiments, molecules, models, datasets, and
researchers. The difficult bugs often enter before an algorithm runs:

- a JSON list is assumed to be a tuple;
- a CSV number remains a string;
- an edge names a node that does not exist;
- a duplicate identifier silently replaces an earlier record;
- a misspelled predicate creates a new relationship type; or
- an output script overwrites its evidence.

Type hints help tools and readers, but Python does not enforce them at runtime. Boundary validation
is still our responsibility.

## The data contract

The course-authored dataset deliberately uses two formats:

| File | Natural shape | Role in the graph |
| --- | --- | --- |
| `scientific_concepts.json` | one nested document containing lists | node metadata |
| `scientific_relationships.csv` | uniform rows and columns | directed edges |

Each edge is a labeled triple—`source`, `predicate`, `target`—plus confidence and an evidence note.
Here, confidence is an editorial teaching value, **not** an empirical probability. The graph is a
compact learning artifact, not an authoritative scientific ontology.

## Professional practice: separate responsibilities

| Layer | Responsibility | Example failure |
| --- | --- | --- |
| transport | find, open, decode | wrong path or encoding |
| syntax | turn text into built-ins | malformed JSON or CSV quoting |
| schema | check fields and container shapes | missing `target_id` |
| domain | create meaningful objects | confidence outside `[0, 1]` |
| aggregate | enforce cross-record rules | edge points to absent node |
| algorithm | answer graph questions | no directed path exists |

Keeping these layers separate makes failures local, tests precise, and code reusable.

## 1. Locate the repository instead of assuming a working directory

VS Code may start a notebook from the repository, the notebook directory, or another configured
folder. Notebook cells do not normally have `__file__`, so search upward for project metadata.
`Path` handles Windows, macOS, and Linux separators.

In [ ]:
import csv
import json
from collections import deque
from collections.abc import Mapping
from dataclasses import asdict, dataclass
from pathlib import Path
from tempfile import TemporaryDirectory

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest Rice DSM project directory.

    Parameters
    ----------
    start : Path
        Directory from which to search upward.

    Returns
    -------
    Path
        Directory containing the course ``pyproject.toml``.

    Raises
    ------
    FileNotFoundError
        If the project cannot be found.
    """

    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        project_file = candidate / "pyproject.toml"
        if project_file.is_file() and "rice-dsm" in project_file.read_text(
            encoding="utf-8"
        ):
            return candidate
    raise FileNotFoundError(f"rice-dsm project not found above {resolved_start}")


project_root = find_project_root(Path.cwd())
data_directory = (
    project_root / "notebooks" / "lecture-02-python-foundations-ii" / "data"
)
concepts_path = data_directory / "scientific_concepts.json"
relationships_path = data_directory / "scientific_relationships.csv"

assert concepts_path.is_file()
assert relationships_path.is_file()

### Inspect before parsing

A suffix is a convention, not proof. Inspect file size and a short text preview before assuming the
schema. Always provide an encoding when opening text; UTF-8 is the course data contract.

In [ ]:
for path in (concepts_path, relationships_path):
    preview = path.read_text(encoding="utf-8")[:90].replace("\n", " ")
    print(f"{path.name}: {path.stat().st_size:,} bytes")
    print(f"  {preview}...")

## 2. JSON: nested structure and native scalar types

`json.load(handle)` reads a text stream and returns ordinary Python values:

- JSON object → `dict`
- JSON array → `list`
- string → `str`
- number → `int` or `float`
- `true` / `false` → `True` / `False`
- `null` → `None`

Parsing proves only that the JSON syntax is valid. It does not prove that this is our expected
document or that its scientific content is sound.

In [ ]:
with concepts_path.open(mode="r", encoding="utf-8") as handle:
    raw_document = json.load(handle)

print(type(raw_document).__name__, raw_document.keys())
print(type(raw_document["concepts"]).__name__, len(raw_document["concepts"]))
print(raw_document["concepts"][0])

assert raw_document["schema_version"] == 1
assert len(raw_document["concepts"]) == 20

### A dictionary is not yet a concept

Raw mappings are flexible but easy to misuse: keys can be absent, values can have the wrong type,
and spelling mistakes appear only at runtime. We will convert every mapping into an immutable
`KnowledgeNode`.

The class owns **single-node invariants**. Cross-record rules belong to the graph later.

In [ ]:
def is_identifier(value: str) -> bool:
    """Return whether a string follows this graph's identifier policy.

    Parameters
    ----------
    value : str
        Candidate identifier.

    Returns
    -------
    bool
        ``True`` for a lowercase Python-style identifier.
    """

    return bool(value) and value.isidentifier() and value == value.lower()


@dataclass(frozen=True, slots=True)
class KnowledgeNode:
    """A validated concept in the scientific knowledge graph.

    Parameters
    ----------
    identifier : str
        Stable lowercase identifier.
    label : str
        Human-readable concept name.
    category : str
        Broad teaching category.
    description : str
        Concise explanation of the concept.
    aliases : tuple of str, optional
        Alternative names.

    Raises
    ------
    TypeError
        If a field has the wrong runtime type.
    ValueError
        If a string is empty or the identifier policy is violated.
    """

    identifier: str
    label: str
    category: str
    description: str
    aliases: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        """Validate one node after dataclass initialization."""

        text_fields = (self.identifier, self.label, self.category, self.description)
        if not all(isinstance(value, str) for value in text_fields):
            raise TypeError("node text fields must be strings")
        if not is_identifier(self.identifier):
            raise ValueError(f"invalid node identifier: {self.identifier!r}")
        if not all(value.strip() for value in text_fields[1:]):
            raise ValueError("node label, category, and description cannot be blank")
        if not isinstance(self.aliases, tuple) or not all(
            isinstance(alias, str) and alias.strip() for alias in self.aliases
        ):
            raise TypeError("aliases must be a tuple of nonblank strings")

### Adapter functions isolate file schema from domain design

JSON gave us `aliases` as a mutable list, while our object promises an immutable tuple. The adapter
performs that conversion explicitly. Avoid `KnowledgeNode(**record)` at an external boundary: it
hides field mapping and produces less intentional errors when a schema changes.

In [ ]:
def require_string(record: Mapping[str, object], key: str) -> str:
    """Read one required, nonblank string field.

    Parameters
    ----------
    record : mapping of str to object
        Parsed external record.
    key : str
        Required field name.

    Returns
    -------
    str
        Validated string value.

    Raises
    ------
    ValueError
        If the field is missing, non-string, or blank.
    """

    value = record.get(key)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{key!r} must be a nonblank string")
    return value


def node_from_mapping(record: Mapping[str, object]) -> KnowledgeNode:
    """Convert a parsed JSON mapping into a knowledge node.

    Parameters
    ----------
    record : mapping of str to object
        One concept from the JSON document.

    Returns
    -------
    KnowledgeNode
        Validated immutable node.

    Raises
    ------
    ValueError
        If a required field or alias violates the schema.
    """

    raw_aliases = record.get("aliases", [])
    if not isinstance(raw_aliases, list) or not all(
        isinstance(alias, str) for alias in raw_aliases
    ):
        raise ValueError("'aliases' must be a list of strings")
    return KnowledgeNode(
        identifier=require_string(record, "id"),
        label=require_string(record, "label"),
        category=require_string(record, "category"),
        description=require_string(record, "description"),
        aliases=tuple(raw_aliases),
    )

In [ ]:
raw_concepts = raw_document.get("concepts")
if not isinstance(raw_concepts, list):
    raise ValueError("'concepts' must be a JSON array")

nodes: list[KnowledgeNode] = []
for index, raw_concept in enumerate(raw_concepts):
    if not isinstance(raw_concept, dict):
        raise ValueError(f"concepts[{index}] must be a JSON object")
    try:
        nodes.append(node_from_mapping(raw_concept))
    except (TypeError, ValueError) as error:
        raise ValueError(f"{concepts_path.name}: concepts[{index}]: {error}") from error

print(nodes[0])
print(nodes[-1])
assert len(nodes) == 20
assert all(isinstance(node, KnowledgeNode) for node in nodes)

## 3. CSV: every field begins as text

CSV is a family of dialects, not “split each line on commas.” Quoted evidence could itself contain a
comma. `csv.DictReader` handles quoting and maps headers to row values.

Open CSV files with `newline=""` so the CSV module—not the platform's text translation—controls
newlines correctly. Unlike JSON, CSV does not encode numeric or Boolean types: even `0.95` arrives
as the string `"0.95"`.

In [ ]:
with relationships_path.open(mode="r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    headers = reader.fieldnames
    first_raw_relationship = next(reader)

print(headers)
print(first_raw_relationship)
print(type(first_raw_relationship["confidence"]).__name__)

expected_headers = ["source_id", "predicate", "target_id", "confidence", "evidence"]
assert headers == expected_headers
assert isinstance(first_raw_relationship["confidence"], str)

### Model a relationship and make conversion policy visible

The parser converts confidence with `float`. The domain object then checks finiteness and range. A
predicate follows an uppercase `RELATION_NAME` vocabulary so an accidental `uses` does not silently
become a second relationship type.

In [ ]:
@dataclass(frozen=True, slots=True)
class Relationship:
    """A directed, labeled fact connecting two node identifiers.

    Parameters
    ----------
    source_id : str
        Identifier where the directed edge begins.
    predicate : str
        Uppercase relationship label.
    target_id : str
        Identifier where the directed edge ends.
    confidence : float
        Editorial teaching confidence in the closed interval ``[0, 1]``.
    evidence : str
        Brief rationale for including the relationship.

    Raises
    ------
    TypeError
        If a field has the wrong runtime type.
    ValueError
        If an identifier, predicate, confidence, or evidence value is invalid.
    """

    source_id: str
    predicate: str
    target_id: str
    confidence: float
    evidence: str

    def __post_init__(self) -> None:
        """Validate one relationship after initialization."""

        if not all(
            isinstance(value, str)
            for value in (self.source_id, self.predicate, self.target_id, self.evidence)
        ):
            raise TypeError("relationship text fields must be strings")
        if not is_identifier(self.source_id) or not is_identifier(self.target_id):
            raise ValueError("relationship endpoints must be valid node identifiers")
        predicate_is_upper = self.predicate == self.predicate.upper()
        if not self.predicate.isidentifier() or not predicate_is_upper:
            raise ValueError("predicate must use UPPER_SNAKE_CASE")
        if not isinstance(self.confidence, float):
            raise TypeError("confidence must be a float")
        if not 0.0 <= self.confidence <= 1.0:
            raise ValueError("confidence must be between 0 and 1")
        if not self.evidence.strip():
            raise ValueError("evidence cannot be blank")

In [ ]:
def relationship_from_mapping(record: Mapping[str, object]) -> Relationship:
    """Convert one parsed CSV row into a relationship.

    Parameters
    ----------
    record : mapping of str to object
        Row produced by ``csv.DictReader``.

    Returns
    -------
    Relationship
        Validated immutable relationship.

    Raises
    ------
    ValueError
        If a field is missing or cannot be converted.
    """

    confidence_text = require_string(record, "confidence")
    try:
        confidence = float(confidence_text)
    except ValueError as error:
        raise ValueError("'confidence' must contain a decimal number") from error

    return Relationship(
        source_id=require_string(record, "source_id"),
        predicate=require_string(record, "predicate"),
        target_id=require_string(record, "target_id"),
        confidence=confidence,
        evidence=require_string(record, "evidence"),
    )

In [ ]:
relationships: list[Relationship] = []
with relationships_path.open(mode="r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    if reader.fieldnames != expected_headers:
        raise ValueError(
            f"{relationships_path.name}: expected headers {expected_headers}, "
            f"received {reader.fieldnames}"
        )
    for line_number, row in enumerate(reader, start=2):
        try:
            relationships.append(relationship_from_mapping(row))
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"{relationships_path.name}: line {line_number}: {error}"
            ) from error

print(relationships[0])
assert len(relationships) == 25
assert all(isinstance(edge, Relationship) for edge in relationships)

## 4. Build a custom `KnowledgeGraph`

Our graph owns aggregate invariants that no individual record can check:

1. node identifiers are unique;
2. relationship endpoints already exist; and
3. an identical `(source, predicate, target)` triple appears at most once.

The class keeps dictionaries private and exposes immutable tuples. This prevents callers from
silently corrupting internal state.

In [ ]:
class KnowledgeGraph:
    """A directed scientific knowledge graph with validated nodes and edges.

    Notes
    -----
    Nodes are indexed by stable identifier. Outgoing relationships are stored as
    adjacency lists, making neighborhood queries proportional to a node's outgoing
    degree rather than the total number of graph relationships.
    """

    def __init__(self) -> None:
        """Create an empty graph."""

        self._nodes: dict[str, KnowledgeNode] = {}
        self._outgoing: dict[str, list[Relationship]] = {}
        self._triples: set[tuple[str, str, str]] = set()

    def __len__(self) -> int:
        """Return the number of nodes."""

        return len(self._nodes)

    def __contains__(self, node_id: object) -> bool:
        """Return whether a node identifier is present."""

        return node_id in self._nodes

    @property
    def nodes(self) -> tuple[KnowledgeNode, ...]:
        """Return an immutable snapshot of nodes in insertion order."""

        return tuple(self._nodes.values())

    @property
    def relationships(self) -> tuple[Relationship, ...]:
        """Return an immutable snapshot of relationships in insertion order."""

        return tuple(
            edge for edges in self._outgoing.values() for edge in edges
        )

    def add_node(self, node: KnowledgeNode) -> None:
        """Add one unique node.

        Parameters
        ----------
        node : KnowledgeNode
            Validated node to add.

        Raises
        ------
        TypeError
            If ``node`` is not a ``KnowledgeNode``.
        ValueError
            If its identifier is already present.
        """

        if not isinstance(node, KnowledgeNode):
            raise TypeError("node must be a KnowledgeNode")
        if node.identifier in self._nodes:
            raise ValueError(f"duplicate node identifier: {node.identifier!r}")
        self._nodes[node.identifier] = node
        self._outgoing[node.identifier] = []

    def add_relationship(self, relationship: Relationship) -> None:
        """Add one relationship after checking graph-level invariants.

        Parameters
        ----------
        relationship : Relationship
            Validated directed relationship.

        Raises
        ------
        TypeError
            If the value is not a ``Relationship``.
        ValueError
            If an endpoint is absent or the triple is duplicated.
        """

        if not isinstance(relationship, Relationship):
            raise TypeError("relationship must be a Relationship")
        missing = [
            node_id
            for node_id in (relationship.source_id, relationship.target_id)
            if node_id not in self._nodes
        ]
        if missing:
            raise ValueError(f"relationship references unknown nodes: {missing}")
        triple = (
            relationship.source_id,
            relationship.predicate,
            relationship.target_id,
        )
        if triple in self._triples:
            raise ValueError(f"duplicate relationship triple: {triple}")
        self._outgoing[relationship.source_id].append(relationship)
        self._triples.add(triple)

### Add nodes before edges

Order is part of the loading algorithm. If we add an edge first, referential-integrity checks must
fail because its endpoints are not known. Once every node is present, each relationship can be
checked immediately.

In [ ]:
graph = KnowledgeGraph()
for node in nodes:
    graph.add_node(node)
for relationship in relationships:
    graph.add_relationship(relationship)

print(f"{len(graph)} nodes; {len(graph.relationships)} relationships")
assert len(graph) == 20
assert len(graph.relationships) == 25
assert "diffusion_model" in graph

### Worked example: failures become teaching tools

An edge can be a perfectly valid `Relationship` and still be invalid **for this graph**. That is why
referential integrity belongs in `KnowledgeGraph.add_relationship`, not in the dataclass.

In [ ]:
unknown_edge = Relationship(
    source_id="diffusion_model",
    predicate="USES",
    target_id="missing_concept",
    confidence=0.9,
    evidence="This endpoint was intentionally omitted.",
)

try:
    graph.add_relationship(unknown_edge)
except ValueError as error:
    print(f"Rejected as designed: {error}")
else:
    raise AssertionError("an edge with an unknown endpoint should be rejected")

## 5. Give the graph a query interface

Clients should ask questions rather than inspect `_outgoing`. A leading underscore communicates
that an attribute is an implementation detail. We now add retrieval and neighborhood behavior to
the class. In a package, these methods would be written in the original class body; the short
subclass lets this notebook reveal the design incrementally.

In [ ]:
class QueryableKnowledgeGraph(KnowledgeGraph):
    """A knowledge graph supporting node, edge, and path queries."""

    def get_node(self, node_id: str) -> KnowledgeNode:
        """Return a node by identifier.

        Parameters
        ----------
        node_id : str
            Stable node identifier.

        Returns
        -------
        KnowledgeNode
            Matching node.

        Raises
        ------
        KeyError
            If no matching node exists.
        """

        try:
            return self._nodes[node_id]
        except KeyError as error:
            raise KeyError(f"unknown node: {node_id!r}") from error

    def relationships_from(
        self,
        node_id: str,
        *,
        predicate: str | None = None,
    ) -> tuple[Relationship, ...]:
        """Return outgoing relationships, optionally filtered by predicate.

        Parameters
        ----------
        node_id : str
            Source node identifier.
        predicate : str or None, optional
            Exact relationship label to retain.

        Returns
        -------
        tuple of Relationship
            Matching relationships in insertion order.
        """

        self.get_node(node_id)
        edges = self._outgoing[node_id]
        if predicate is None:
            return tuple(edges)
        return tuple(edge for edge in edges if edge.predicate == predicate)

    def neighbors(
        self,
        node_id: str,
        *,
        predicate: str | None = None,
    ) -> tuple[KnowledgeNode, ...]:
        """Return target nodes reached by outgoing relationships.

        Parameters
        ----------
        node_id : str
            Source node identifier.
        predicate : str or None, optional
            Exact relationship label to retain.

        Returns
        -------
        tuple of KnowledgeNode
            Target nodes, with duplicates removed in edge order.
        """

        seen: set[str] = set()
        result: list[KnowledgeNode] = []
        for edge in self.relationships_from(node_id, predicate=predicate):
            if edge.target_id not in seen:
                result.append(self._nodes[edge.target_id])
                seen.add(edge.target_id)
        return tuple(result)

In [ ]:
query_graph = QueryableKnowledgeGraph()
for node in nodes:
    query_graph.add_node(node)
for relationship in relationships:
    query_graph.add_relationship(relationship)

diffusion_neighbors = query_graph.neighbors("diffusion_model", predicate="USES")
for neighbor in diffusion_neighbors:
    print(f"Diffusion model USES {neighbor.label}")

assert {node.identifier for node in diffusion_neighbors} == {
    "markov_chain",
    "differential_equation",
    "neural_network",
}

## 6. Path finding with breadth-first search

“How is a diffusion model connected to a scientific measurement?” is a graph question. Breadth-first
search (BFS) explores all paths of length 1, then length 2, and so on. In an unweighted graph, the
first discovered route uses the fewest edges.

The queue stores complete paths for clarity. A larger system would usually store one predecessor per
visited node to reduce memory use.

In [ ]:
class PathKnowledgeGraph(QueryableKnowledgeGraph):
    """A queryable knowledge graph with directed breadth-first search."""

    def find_path(self, start_id: str, end_id: str) -> tuple[str, ...] | None:
        """Find a shortest directed path between two nodes.

        Parameters
        ----------
        start_id : str
            Identifier at which traversal begins.
        end_id : str
            Desired destination identifier.

        Returns
        -------
        tuple of str or None
            Node identifiers along a shortest path, or ``None`` if unreachable.

        Raises
        ------
        KeyError
            If either endpoint is unknown.
        """

        self.get_node(start_id)
        self.get_node(end_id)
        queue: deque[tuple[str, ...]] = deque([(start_id,)])
        visited = {start_id}

        while queue:
            path = queue.popleft()
            current = path[-1]
            if current == end_id:
                return path
            for neighbor in self.neighbors(current):
                if neighbor.identifier not in visited:
                    visited.add(neighbor.identifier)
                    queue.append((*path, neighbor.identifier))
        return None

In [ ]:
science_graph = PathKnowledgeGraph()
for node in nodes:
    science_graph.add_node(node)
for relationship in relationships:
    science_graph.add_relationship(relationship)

path = science_graph.find_path("diffusion_model", "scientific_measurement")
assert path is not None
print(" → ".join(science_graph.get_node(node_id).label for node_id in path))

assert path == (
    "diffusion_model",
    "markov_chain",
    "probability_distribution",
    "scientific_measurement",
)

### Read paths as hypotheses, not proofs

The path above says our dataset contains three directed assertions. It does not prove that a
diffusion model is a measurement theory, nor that every use of a Markov chain has the same meaning.

Knowledge graphs compress context. Industry systems therefore track provenance, qualifiers,
timestamps, source documents, and competing claims. A graph query returns what was encoded—not
automatic truth.

## 7. Package the loading pipeline

Notebook variables are convenient during exploration; reusable software needs a function with a
clear interface. This loader performs transport, syntax, schema, domain, and aggregate validation in
one public operation while delegating each conversion to the smaller functions we already tested.

In [ ]:
def load_knowledge_graph(
    concepts_file: Path,
    relationships_file: Path,
) -> PathKnowledgeGraph:
    """Load a validated scientific knowledge graph from JSON and CSV.

    Parameters
    ----------
    concepts_file : Path
        JSON document containing a ``concepts`` array.
    relationships_file : Path
        CSV file containing the exact relationship schema.

    Returns
    -------
    PathKnowledgeGraph
        Fully validated graph.

    Raises
    ------
    OSError
        If either file cannot be read.
    json.JSONDecodeError
        If the JSON syntax is invalid.
    ValueError
        If a schema or graph invariant is violated.
    """

    with concepts_file.open(mode="r", encoding="utf-8") as handle:
        document = json.load(handle)
    raw_nodes = document.get("concepts")
    if document.get("schema_version") != 1 or not isinstance(raw_nodes, list):
        raise ValueError(f"{concepts_file.name}: unsupported concept schema")

    loaded_graph = PathKnowledgeGraph()
    for index, raw_node in enumerate(raw_nodes):
        if not isinstance(raw_node, dict):
            message = f"{concepts_file.name}: concepts[{index}] must be an object"
            raise ValueError(message)
        try:
            loaded_graph.add_node(node_from_mapping(raw_node))
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"{concepts_file.name}: concepts[{index}]: {error}"
            ) from error

    with relationships_file.open(
        mode="r", encoding="utf-8", newline=""
    ) as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != expected_headers:
            raise ValueError(f"{relationships_file.name}: unexpected CSV headers")
        for line_number, row in enumerate(reader, start=2):
            try:
                loaded_graph.add_relationship(relationship_from_mapping(row))
            except (TypeError, ValueError) as error:
                raise ValueError(
                    f"{relationships_file.name}: line {line_number}: {error}"
                ) from error

    return loaded_graph


loaded_graph = load_knowledge_graph(concepts_path, relationships_path)
assert len(loaded_graph) == 20
assert len(loaded_graph.relationships) == 25

## 8. Derived serialization: preserve the evidence

Never overwrite raw inputs during exploration. The next function creates a JSON-compatible snapshot
of the graph. Tuples become JSON arrays when serialized. A production export would also include a
schema version, source hashes, creation time, and the exact code version.

In [ ]:
def graph_to_mapping(graph: KnowledgeGraph) -> dict[str, object]:
    """Convert a graph into JSON-serializable built-ins.

    Parameters
    ----------
    graph : KnowledgeGraph
        Graph to serialize.

    Returns
    -------
    dict of str to object
        Versioned mapping containing node and relationship records.
    """

    return {
        "schema_version": 1,
        "nodes": [asdict(node) for node in graph.nodes],
        "relationships": [asdict(edge) for edge in graph.relationships],
    }


with TemporaryDirectory() as temporary_directory:
    output_path = Path(temporary_directory) / "knowledge_graph_snapshot.json"
    with output_path.open(mode="w", encoding="utf-8") as handle:
        json.dump(graph_to_mapping(loaded_graph), handle, indent=2, sort_keys=True)
        handle.write("\n")
    round_trip = json.loads(output_path.read_text(encoding="utf-8"))
    print(output_path.name, output_path.stat().st_size, "bytes")

assert len(round_trip["nodes"]) == len(loaded_graph)
assert len(round_trip["relationships"]) == len(loaded_graph.relationships)

## Debugging guide

When loading fails, locate the layer before changing code:

| Symptom | Likely layer | First check |
| --- | --- | --- |
| `FileNotFoundError` | transport | `Path.cwd()` and resolved path |
| `UnicodeDecodeError` | transport | documented encoding and original bytes |
| `JSONDecodeError` | syntax | reported line and column |
| unexpected CSV headers | schema | delimiter, header spelling, file version |
| `float(...)` failure | schema/conversion | raw confidence text |
| invalid identifier | domain | identifier policy |
| unknown endpoint | aggregate | node file and load order |
| `None` from `find_path` | algorithm/data | edge direction and reachability |

Do not “fix” failures by ignoring decoding errors, inventing missing nodes, or swallowing broad
exceptions. Those responses erase evidence.

## Common failure: trusting type hints at runtime

The annotation `confidence: float` does not stop a caller from passing a string. Our explicit
`__post_init__` check does. Parsers and constructors have different jobs: the parser converts
`"0.95"`; the constructor refuses anything outside its runtime contract.

In [ ]:
try:
    Relationship(
        source_id="vector",
        predicate="IS_A",
        target_id="matrix",
        confidence="high",  # type: ignore[arg-type]
        evidence="Intentional bad input for the lesson.",
    )
except TypeError as error:
    print(f"Rejected as designed: {error}")
else:
    raise AssertionError("a string confidence should be rejected")

## Guided exercise: summarize the relationship vocabulary

Count each predicate without importing `Counter`. Then identify the most frequent predicate. Think
about whether frequency means scientific importance—it does not.

In [ ]:
predicate_counts: dict[str, int] = {}
for relationship in loaded_graph.relationships:
    predicate_counts[relationship.predicate] = (
        predicate_counts.get(relationship.predicate, 0) + 1
    )

most_frequent_predicate = max(predicate_counts, key=predicate_counts.get)  # type: ignore[arg-type]
print(predicate_counts)
print("most frequent:", most_frequent_predicate)

assert predicate_counts["USES"] == 5

## Independent exercise: category subgraph

Write `nodes_in_category(graph, category)` with NumPy-style documentation and type hints. Return an
immutable tuple sorted by label. Test the happy path and an absent category. Decide whether absence
should return an empty tuple or raise; document your policy.

In [ ]:
def nodes_in_category(
    graph: KnowledgeGraph,
    category: str,
) -> tuple[KnowledgeNode, ...]:
    """Return nodes in one category, sorted by label.

    Parameters
    ----------
    graph : KnowledgeGraph
        Graph to query.
    category : str
        Exact category name.

    Returns
    -------
    tuple of KnowledgeNode
        Matching nodes sorted by display label; empty when no category matches.
    """

    return tuple(sorted(
        (node for node in graph.nodes if node.category == category),
        key=lambda node: node.label,
    ))


probability_nodes = nodes_in_category(loaded_graph, "probability")
print([node.label for node in probability_nodes])
assert len(probability_nodes) == 5
assert nodes_in_category(loaded_graph, "astronomy") == ()

## Extension: make provenance first-class

For a research-grade graph, redesign `Relationship` so evidence is not a free-text sentence alone.
Introduce a `SourceReference` object with a persistent identifier, title, version, and locator.

Then consider difficult cases:

- two papers disagree about a relationship;
- a claim is valid only under stated assumptions;
- a source is retracted;
- confidence is produced by a model rather than a human curator; and
- a relationship changes through time.

This is where data modeling becomes epistemology: the schema controls which kinds of knowledge and
uncertainty the system is capable of representing.

## Retrieval practice

Without running code, answer:

1. Why is valid JSON not necessarily valid course data?
2. Why does CSV confidence require explicit conversion while JSON numbers may not?
3. Which invariant belongs to `Relationship`, and which belongs to `KnowledgeGraph`?
4. Why are nodes loaded before relationships?
5. What does BFS guarantee here, and what does it not guarantee scientifically?
6. Why do query methods return tuples instead of internal lists?
7. What information is missing before this graph could support research claims?

## Takeaway

You built a real software boundary and a small graph system:

```text
JSON concepts ──parse + validate──┐
                                  ├──> KnowledgeGraph ──> neighbors, paths, export
CSV relationships ─parse + validate┘
```

The important habits are durable:

- use standard parsers instead of string tricks;
- convert external records explicitly;
- validate local rules in value objects and cross-record rules in aggregates;
- expose behavior through a small documented interface;
- preserve raw evidence and record provenance; and
- interpret computational results no more strongly than the data permits.

## Further reading

- Python documentation: **Input and Output**, including file objects and JSON
- Python documentation: **`pathlib` — Object-oriented filesystem paths**
- Python documentation: **`csv` — CSV File Reading and Writing**
- Python documentation: **`json` — JSON encoder and decoder**
- Python documentation: **`collections.deque`**, used for efficient BFS queues
- Course data README: provenance and limitations of this teaching graph

Next: move these foundations into modules, packages, and automated tests in Lecture 3.